# SWaT Unconstrained Cache Generation: Ultra-VETime Multi-Stage Teacher
### High-Density Curriculum Caching & Dedicated Head-to-Head Evaluation Cache
- **Backbone**: `facebook/dinov2-large` (1024-dim self-supervised visual patch features)
- **Multi-Regime Operational Centroids**: $K=5$ KMeans nominal operating clusters per stage
- **Multi-Stage Fusion**: Top-2 physical plant stages evaluated per window
- **Rendering**: Fast direct Agg canvas buffer copy (zero PNG compression bottleneck)
- **Multi-GPU**: Shards candidate windows across `cuda:0` and `cuda:1` if dual GPUs are provisioned

In [ ]:
# --- Step 1: Workspace Setup & Codebase Unpacking ---
import os, sys, shutil, subprocess, base64, io, zipfile

repo_dir = '/kaggle/working/VisTAD'
os.makedirs(repo_dir, exist_ok=True)
os.chdir(repo_dir)

print('Unpacking self-contained VisTAD codebase (src, vlm4ts, configs)...')
b64_bundle = "..."
raw_zip = base64.b64decode(b64_bundle)
with zipfile.ZipFile(io.BytesIO(raw_zip), 'r') as z:
    z.extractall(repo_dir)
print('VisTAD codebase unpacked successfully.')
print('Extracted files:', os.listdir(repo_dir))


In [ ]:
# --- Step 2: GPU Accelerator & Multi-GPU Diagnostics ---
import torch
print(f'PyTorch Version: {torch.__version__}')
print(f'CUDA Available: {torch.cuda.is_available()}')
num_gpus = torch.cuda.device_count()
print(f'Detected {num_gpus} GPU(s):')
for i in range(num_gpus):
    props = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {props.name} ({props.total_memory / (1024**3):.1f} GB VRAM)')


In [ ]:
# --- Step 3: SWaT Dataset Auto-Discovery ---
import os, glob, shutil
os.makedirs('data/swat', exist_ok=True)

dataset_paths = glob.glob('/kaggle/input/**/SWaT_Dataset_*.csv', recursive=True)
print(f'Found {len(dataset_paths)} SWaT dataset candidates in /kaggle/input:')
for p in dataset_paths:
    fname = os.path.basename(p)
    target = os.path.join('data/swat', fname)
    if not os.path.exists(target):
        print(f'  Linking {p} -> {target}')
        try:
            os.symlink(p, target)
        except OSError:
            shutil.copy2(p, target)

print('SWaT data directory contents:', os.listdir('data/swat'))


In [ ]:
# --- Step 4: Execute SWaT Unconstrained Cache Generation ---
import os, subprocess, time
os.environ['PYTHONPATH'] = '.'

cmd = [
    'python', 'vlm4ts/generate_cache.py',
    '--data_dir', 'data/swat',
    '--output_train_cache', 'cache/swat_train_ultravetime_cache.pt',
    '--output_eval_cache', 'cache/swat_eval_head_to_head_cache.pt',
    '--window_size', '200',
    '--num_train_windows', '3500',
    '--num_eval_windows', '1200',
    '--model_id', 'facebook/dinov2-large',
    '--n_regimes', '5'
]
print('Executing unconstrained caching command:', ' '.join(cmd))
t0 = time.time()
proc = subprocess.run(cmd, check=True)
duration = time.time() - t0
print(f'Caching completed successfully in {duration:.1f}s ({duration/60:.2f} min).')


In [ ]:
# --- Step 5: Verify Cache Statistics & Ground Truth Distributions ---
import torch, os
import numpy as np

for cache_path, label in [
    ('cache/swat_train_ultravetime_cache.pt', 'TRAINING CURRICULUM CACHE'),
    ('cache/swat_eval_head_to_head_cache.pt', 'HEAD-TO-HEAD EVALUATION CACHE')
]:
    if not os.path.exists(cache_path):
        print(f'WARNING: {cache_path} not found!')
        continue
    size_mb = os.path.getsize(cache_path) / (1024 * 1024)
    c = torch.load(cache_path, weights_only=False)
    windows = c['windows']
    flags = [w['y_flag'] for w in windows.values()]
    scores = [w['teacher_score'] for w in windows.values()]
    anom_count = sum(1 for f in flags if f > 0.5)
    norm_count = len(flags) - anom_count
    print(f'\n================== {label} ==================')
    print(f'File Size: {size_mb:.2f} MB | Total Windows: {len(windows)}')
    print(f'Attack Windows: {anom_count} ({anom_count/len(windows)*100:.1f}%) | Nominal: {norm_count}')
    print(f'Teacher Anomaly Score: min={min(scores):.4f}, median={np.median(scores):.4f}, max={max(scores):.4f}')
    first_w = next(iter(windows.values()))
    print(f'Telemetry Tensor x: {tuple(first_w["x"].shape)}')
    print(f'Feature Target z_teacher: {tuple(first_w["z_teacher"].shape)}')
    print(f'Temporal Mask teacher_dense_mask: {tuple(first_w["teacher_dense_mask"].shape)}')


In [ ]:
# --- Step 6: Package Output Bundle for Retrieval ---
import os, zipfile, glob

bundle_path = '/kaggle/working/swat_ultravetime_cache_bundle.zip'
print(f'Packaging cache files into {bundle_path}...')
with zipfile.ZipFile(bundle_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob('cache/*.pt'):
        print(f'  Adding {f} ({os.path.getsize(f)/(1024*1024):.2f} MB)')
        z.write(f, os.path.relpath(f, '.'))
print(f'Bundle created successfully ({os.path.getsize(bundle_path)/(1024*1024):.2f} MB).')
print('Output directory contents:', os.listdir('/kaggle/working'))
